# 02 — Calculate ELO

Loads cached raw match data, runs the ELO pipeline, writes results to `data/elo.db`.
Idempotent — re-running skips already-processed matches.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
RAW_DIR = Path('../data/raw')
DB_PATH = str(Path('../data/elo.db'))

In [ ]:
from src.db import create_db

conn = create_db(DB_PATH)
print(f'DB initialized at {DB_PATH}')

In [ ]:
all_matches = []
for f in sorted(RAW_DIR.glob('*.json')):
    with open(f) as fh:
        all_matches.extend(json.load(fh))

all_matches.sort(key=lambda m: m['date'])
print(f'Loaded {len(all_matches)} matches')
print(f'Date range: {all_matches[0]["date"]} — {all_matches[-1]["date"]}')

In [ ]:
from src.db import (
    upsert_player, insert_match, insert_xg_event,
    upsert_player_rating, get_current_ratings
)
from src.elo import process_match, BASELINE_RATING

ratings = get_current_ratings(conn)
already_processed = {
    row[0] for row in conn.execute('SELECT match_id FROM matches').fetchall()
}

skipped = 0
for match in all_matches:
    match_id = match['match_id']
    if match_id in already_processed:
        skipped += 1
        continue

    home_team = match['home_team']
    away_team = match['away_team']
    home_team_id = match['home_team_id']
    away_team_id = match['away_team_id']

    starters = match['starters']
    home_starters = starters.get(home_team, [])
    away_starters = starters.get(away_team, [])
    subs = match['substitutions']

    insert_match(conn, match_id, match['date'], home_team, away_team,
                 match['season'], match['source'])
    for pid, name in match['player_names'].items():
        upsert_player(conn, pid, name)
    for shot in match['shots']:
        insert_xg_event(
            conn, shot['event_id'], match_id, shot['minute'],
            shot['xg'], shot['shooting_team_id'],
            shot.get('shooter_player_id')
        )

    updated_ratings, deltas = process_match(
        ratings=ratings,
        home_team_id=home_team_id,
        away_team_id=away_team_id,
        home_starters=home_starters,
        away_starters=away_starters,
        substitutions=subs,
        xg_events=match['shots']
    )

    all_pids = set(home_starters + away_starters + [s['player_on_id'] for s in subs])
    for pid in all_pids:
        team = home_team if (
            pid in home_starters or
            any(s['player_on_id'] == pid and s['team_id'] == home_team_id for s in subs)
        ) else away_team
        upsert_player_rating(
            conn, pid, match_id, match['date'],
            updated_ratings.get(pid, BASELINE_RATING),
            deltas.get(pid, 0.0),
            team
        )

    ratings = updated_ratings

conn.commit()
print(f'Processed {len(all_matches) - skipped} new matches, skipped {skipped}.')

In [ ]:
top = conn.execute("""
    SELECT p.name, pr.rating_after
    FROM player_ratings pr
    JOIN players p ON pr.player_id = p.player_id
    INNER JOIN (
        SELECT player_id, MAX(date) AS d FROM player_ratings GROUP BY player_id
    ) latest ON pr.player_id = latest.player_id AND pr.date = latest.d
    ORDER BY pr.rating_after DESC LIMIT 10
""").fetchall()

print('Top 10 by current ELO:')
for row in top:
    print(f'  {row[0]}: {row[1]:.1f}')